In [ ]:
public interface ICalculable
{
    void CalculateTotal();
}

public interface ITrackable
{
    void MarkAsPaid();
}

public interface IPayable
{
    void Pay(decimal amount);
}

public class Invoice : ICalculable, ITrackable, IPayable
{
    public string InvoiceNumber { get; protected set; }
    public DateTime IssueDate { get; protected set; }
    public decimal TotalAmount { get; protected set; }

    public string Currency { get; set; } = "EUR";
    public decimal Discount { get; set; }
    public bool IsPaid { get; private set; }
    public string CustomerName { get; set; }

    protected List<LineItem> LineItems { get; } = new();

    public Invoice(string invoiceNumber, DateTime issueDate)
    {
        InvoiceNumber = invoiceNumber;
        IssueDate = issueDate;
    }

    void ICalculable.CalculateTotal()
    {
        TotalAmount = LineItems.Sum(x => x.Total) - Discount;
    }

    void ITrackable.MarkAsPaid()
    {
        IsPaid = true;
    }

    void IPayable.Pay(decimal amount)
    {
        if (amount >= TotalAmount)
        {
            IsPaid = true;
            Console.WriteLine($"Invoice {InvoiceNumber} paid.");
        }
        else
        {
            Console.WriteLine($"Insufficient payment for Invoice {InvoiceNumber}");
        }
    }

    public void ApplyDiscount(decimal value)
    {
        Discount = value;
        CalculateTotal();
    }

    public void PrintInfo()
    {
        Console.WriteLine($"{InvoiceNumber} | {TotalAmount:C} | Paid: {IsPaid}");
    }

    public IReadOnlyList<LineItem> GetLineItems() => LineItems.AsReadOnly();
}

public class GoodsInvoice : Invoice, ICalculable, ITrackable, IPayable
{
    public DateTime SupplyDate { get; set; }
    public decimal DeliveryCost { get; set; }

    public GoodsInvoice(string number, DateTime issueDate, DateTime supplyDate)
        : base(number, issueDate)
    {
        SupplyDate = supplyDate;
    }

    void ICalculable.CalculateTotal()
    {
        base.CalculateTotal();
        TotalAmount += DeliveryCost;
        Console.WriteLine("[GoodsInvoice] Delivery included");
    }

    void ITrackable.MarkAsPaid()
    {
        IsPaid = true;
        Console.WriteLine("[GoodsInvoice] Marked as paid");
    }

    void IPayable.Pay(decimal amount)
    {
        if (amount >= TotalAmount)
        {
            IsPaid = true;
            Console.WriteLine($"GoodsInvoice {InvoiceNumber} paid.");
        }
        else
        {
            Console.WriteLine($"Insufficient payment for GoodsInvoice {InvoiceNumber}");
        }
    }

    public void AddLine(LineItem item, bool fragile)
    {
        if (fragile)
            item.UnitPrice += 5;

        base.AddLine(item);
    }
}

public class LineItem
{
    public string Description { get; set; }
    public decimal Quantity { get; set; }
    public decimal UnitPrice { get; set; }
    public bool IsReturned { get; set; }

    public decimal Discount { get; set; }

    public decimal Total => Quantity * UnitPrice - Discount;

    public LineItem(string desc, decimal qty, decimal price)
    {
        Description = desc;
        Quantity = qty;
        UnitPrice = price;
    }

    public decimal GetTotal(bool includeTax)
    {
        var total = Total;
        return includeTax ? total * 1.2m : total;
    }
}

public class Repository<T>
{
    private readonly List<T> _items = new();

    public void Add(T item) => _items.Add(item);

    public IEnumerable<T> GetAll() => _items;

    public T Find(Func<T, bool> predicate)
    {
        return _items.FirstOrDefault(predicate);
    }
}

public class Processor<T> where T : Invoice
{
    public decimal CalculateTotal(IEnumerable<T> invoices)
    {
        return invoices.Sum(i => i.TotalAmount);
    }

    public void Process(T invoice)
    {
        invoice.CalculateTotal();
        invoice.PrintInfo();
    }
}

public class InvoiceProcessor
{
    private readonly ICalculable _calculableInvoice;
    private readonly ITrackable _trackableInvoice;
    private readonly IPayable _payableInvoice;

    public InvoiceProcessor(ICalculable calculableInvoice, ITrackable trackableInvoice, IPayable payableInvoice)
    {
        _calculableInvoice = calculableInvoice;
        _trackableInvoice = trackableInvoice;
        _payableInvoice = payableInvoice;
    }

    public void ProcessInvoice()
    {
        _calculableInvoice.CalculateTotal();
        _trackableInvoice.MarkAsPaid();
        _payableInvoice.Pay(100);
    }
}

class Program
{
    static void Main()
    {
        var goodsInvoice = new GoodsInvoice("G1", DateTime.Now, DateTime.Now)
        {
            DeliveryCost = 50
        };

        var invoiceProcessor = new InvoiceProcessor(
            (ICalculable)goodsInvoice,
            (ITrackable)goodsInvoice,
            (IPayable)goodsInvoice
        );

        invoiceProcessor.ProcessInvoice();
        goodsInvoice.PrintInfo();
    }
}